In [23]:
from typing import List
from matplotlib import pyplot as plt
import random
import itertools
import tqdm

In [21]:
Vector = List[float]

def argmax(v: list) -> int:
    """Devuelve el índice del valor máximo en la lista v"""
    return v.index(max(v))

# Primero necesitamos poder sumar una lista de vectores
def vector_sum(vectors: List[Vector]) -> Vector:
    """Suma todos los vectores correspondientes"""
    num_elements = len(vectors[0])
    return [sum(v[i] for v in vectors)
            for i in range(num_elements)]

# Ahora definimos el promedio
def vector_mean(vectors: List[Vector]) -> Vector:
    """Calcula el vector cuyo i-ésimo elemento es la media de los
    i-ésimos elementos de los vectores de entrada."""
    n = len(vectors)
    return [x / n for x in vector_sum(vectors)]

def vector_subtract(v: Vector, w: Vector) -> Vector:
    """Resta elementos correspondientes: [v1 - w1, v2 - w2, ...]"""
    return [v_i - w_i for v_i, w_i in zip(v, w)]

def sum_of_squares(v: Vector) -> float:
    """Devuelve v_1 * v_1 + ... + v_n * v_n"""
    return sum(v_i ** 2 for v_i in v)

def squared_distance(v: Vector, w: Vector) -> float:
    """Calcula (v_1 - w_1)**2 + ... + (v_n - w_n)**2"""
    return sum_of_squares(vector_subtract(v, w))

# Creamos puntos alrededor de tres centros específicos
inputs = (
    [[ -26 + random.random(),  -5 + random.random()] for _ in range(30)] +
    [[  18 + random.random(),  20 + random.random()] for _ in range(30)]
)

In [9]:
def num_differences(v1: Vector, v2: Vector) -> int:
    assert len(v1) == len(v2)
    return len([x1 for x1, x2 in zip(v1, v2) if x1 != x2])

assert num_differences([1, 2, 3], [2, 1, 3]) == 2
assert num_differences([1, 2], [1, 2]) == 0

In [ ]:
def cluster_means(k: int, inputs: List[Vector], assignment: List[int]) -> List[Vector]:
    # clusters[i] contiene las entradas cuya asignasion es i
    clusters = [[] for i in range(k)]
    for input, assignment in zip(inputs, assignment):
        clusters[assignment].append(input)
    # si un grupo esta vacio, usa un punto aleatorio
    return [vector_mean(cluster) if cluster else random.choice(inputs) for cluster in clusters]

In [14]:
class KMeans:
    def __init__(self, k: int) -> None:
        self.k = k
        self.means = None
    
    def classify(self, input: Vector) -> int:
        """Devolver el índice del clúster más cercano a la entrada"""
        return min(range(self.k), key=lambda i: squared_distance(input, self.means[i]))
    
    def train(self, inputs: List[Vector]) -> None:
        # inicia con asignaciones aleatorias
        assignments = [random.randrange(self.k) for _ in inputs]
        with tqdm.tqdm(itertools.count()) as t:
            for _ in t:
                # calcula la media y halla nuevas asignaciones
                self.means = cluster_means(self.k, inputs, assignments)
                new_assignments = [self.classify(input) for input in inputs]
                # conprueba cuantas asignaciones cambiaron y si hemos terminado
                num_chaged = num_differences(assignments, new_assignments)
                if num_chaged == 0:
                    return
                # si no mantiene las nuevas asignaciones y calcula nuevas medias
                assignments = new_assignments
                self.means = cluster_means(self.k, inputs, assignments)
                t.set_description(f"changed: {num_chaged} / {len(inputs)}")

# Encuentros

In [18]:
random.seed(12)
clusterer = KMeans(k=3)
clusterer.train(inputs)
means = sorted(clusterer.means)
assert len(means) == 3

# revisa que las medidas esten cerca de lo que esperamos

assert squared_distance(means[0], [-44, 5]) < 1
assert squared_distance(means[1], [-16, -10]) < 1
assert squared_distance(means[2], [18, 20]) < 1

changed: 20 / 60: : 2it [00:00, 383.50it/s]


In [22]:
random.seed(0)
clusterer = KMeans(k=2)
clusterer.train(inputs)
means = sorted(clusterer.means)

assert len(means) == 2
assert squared_distance(means[0], [-26, -5]) < 1
assert squared_distance(means[1], [18, 20]) < 1

changed: 25 / 60: : 1it [00:00, 397.19it/s]


In [25]:
def squared_clustering_errors(inputs: List[Vector], k: int) -> float:
    """Encuentra el error cuadrado total de k-means agrupando las entradas"""
    clusterer = KMeans(k)
    clusterer.train(inputs)
    means = clusterer.means
    assignments = [clusterer.classify(input) for input in inputs]
    return sum(squared_distance(input, means[cluster]) for input, cluster in zip(inputs, assignments))

In [ ]:
# ahora traza desde 1 hasta len(inputs) clusteres

